In [1]:
# Cell 1 - Verificar archivos en Bronze
import json

lakehouses = mssparkutils.lakehouse.list()
for lh in lakehouses:
    if 'Bronze' in lh['displayName']:
        BRONZE_ID = lh['id']
        WORKSPACE_ID = lh['workspaceId']
        print(f"Bronze ID: {BRONZE_ID}")
        print(f"Workspace ID: {WORKSPACE_ID}")

BRONZE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"

schemas = mssparkutils.fs.ls(f"{BRONZE_PATH}/Files")
for s in schemas:
    files = mssparkutils.fs.ls(f"{BRONZE_PATH}/Files/{s.name}")
    print(f"{s.name}: {len(files)} tablas")

StatementMeta(, e482a9e0-1aa9-427d-b69f-95d78e1fbce7, 3, Finished, Available, Finished, False)

Bronze ID: f239c4cb-d96b-476a-8174-e8a857dbb690
Workspace ID: dafbb8c2-74f5-4297-9b91-d604b412950c
HumanResources: 5 tablas
Person: 12 tablas
Production: 23 tablas
Purchasing: 5 tablas
Sales: 19 tablas
dbo: 3 tablas


In [2]:
# Cell 2 - Configurar rutas
lakehouses = mssparkutils.lakehouse.list()
for lh in lakehouses:
    if 'Silver' in lh['displayName']:
        SILVER_ID = lh['id']
    if 'Gold' in lh['displayName']:
        GOLD_ID = lh['id']
    if 'Bronze' in lh['displayName']:
        BRONZE_ID = lh['id']
        WORKSPACE_ID = lh['workspaceId']

BRONZE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"
SILVER_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"
GOLD_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

print(f"Bronze: {BRONZE_ID}")
print(f"Silver: {SILVER_ID}")
print(f"Gold: {GOLD_ID}")
print("Rutas configuradas OK")

StatementMeta(, e482a9e0-1aa9-427d-b69f-95d78e1fbce7, 4, Finished, Available, Finished, False)

Bronze: f239c4cb-d96b-476a-8174-e8a857dbb690
Silver: 7238c65b-dcce-476a-9311-134ae8788a34
Gold: c3518afb-2909-426d-9726-991744a55153
Rutas configuradas OK


In [3]:
# Cell 3 - Ingestar todos los parquet a tablas Delta en Bronze
from pyspark.sql.functions import lit

schemas = mssparkutils.fs.ls(f"{BRONZE_PATH}/Files")

total_ok = 0
total_error = 0

for schema in schemas:
    schema_name = schema.name.rstrip('/')
    files = mssparkutils.fs.ls(f"{BRONZE_PATH}/Files/{schema_name}")
    
    for file in files:
        table_name = file.name.replace('.parquet', '').rstrip('/')
        path = f"{BRONZE_PATH}/Files/{schema_name}/{file.name}"
        
        try:
            df = spark.read.parquet(path)
            df = df.withColumn("_source_schema", lit(schema_name))
            
            delta_table_name = f"bronze_{schema_name}_{table_name}".lower()
            df.write.format("delta").mode("overwrite").saveAsTable(delta_table_name)
            
            print(f"OK: {delta_table_name} ({df.count():,} filas)")
            total_ok += 1
            
        except Exception as e:
            print(f"ERROR: {schema_name}.{table_name} -> {e}")
            total_error += 1

print(f"\nTotal OK: {total_ok} | Total ERROR: {total_error}")

StatementMeta(, e482a9e0-1aa9-427d-b69f-95d78e1fbce7, 5, Finished, Available, Finished, False)

OK: bronze_humanresources_department (16 filas)
OK: bronze_humanresources_employeedepartmenthistory (296 filas)
OK: bronze_humanresources_employeepayhistory (316 filas)
OK: bronze_humanresources_jobcandidate (13 filas)
ERROR: HumanResources.Shift -> An error occurred while calling o5368.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 61.0 failed 4 times, most recent failure: Lost task 0.3 in stage 61.0 (TID 77) (vm-b4820930 executor 2): org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIME(MICROS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1836)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:199)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:276)
	at scala.Option.getOrE